In [2]:
!pip uninstall NN_UTCI -y
!pip uninstall utci-nn -y

Found existing installation: NN_UTCI 0.1.0
Uninstalling NN_UTCI-0.1.0:
  Successfully uninstalled NN_UTCI-0.1.0


In [3]:
!pip install git+https://github.com/bikempastine/utci-nn.git

  Cloning https://github.com/bikempastine/utci-nn.git to c:\users\nerc-user\appdata\local\temp\pip-req-build-d7y8fcgs
  Resolved https://github.com/bikempastine/utci-nn.git to commit 2a62d84732ac9b1189def82b61b0318fc4490119
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for utci-nn: filename=utci_nn-0.1.0-py3-none-any.whl size=44693 sha256=c227c38a4704520655c6f395dea92a21a58e60d5f421a17d2f62e389eeab2e58
  Stored in directory: C:\Users\nerc-user\AppData\Local\Temp\pip-ephem-wheel-cache-pdycldto\wheels\cb\d4\76\f033a7e137bea61aab5827a4d379f7a76e4aec5f728f786ae0
Successfully built utci-nn


  Running command git clone --filter=blob:none --quiet https://github.com/bikempastine/utci-nn.git 'C:\Users\nerc-user\AppData\Local\Temp\pip-req-build-d7y8fcgs'


In [4]:
!pip show utci-nn

Name: utci-nn
Version: 0.1.0
Summary: UTCI neural network approximator by Pastine et al.
Home-page: 
Author: 
Author-email: 
License: 
Location: C:\Users\nerc-user\.conda\envs\utci\Lib\site-packages
Requires: numpy, scikit-learn, torch
Required-by: 


In [1]:
# Scientific stack
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

# External models
from utci_nn import calculate_utci


In [2]:
?calculate_utci

Signature:
calculate_utci(
    Ta: Union[float, numpy.ndarray, pandas.core.series.Series],
    Tr: Union[float, numpy.ndarray, pandas.core.series.Series],
    va: Union[float, numpy.ndarray, pandas.core.series.Series],
    rH: Union[float, numpy.ndarray, pandas.core.series.Series],
    oob: str = 'nan',
) -> numpy.ndarray
Docstring:
Calculate UTCI using a pre-trained neural network approximator as described in Pastine et al.
NN is trained on the same data as the polynomial approximation in Bröde et al. (2012) and covers the same input domain.

Accepts scalars, numpy arrays, or pandas Series for all meterological inputs.

Parameters
----------
Ta  : Air temperature (°C),          valid range: -50 to +50
Tr  : Mean radiant temperature (°C), valid range: Ta-80 to Ta+120
va  : Wind speed at 10 m (m/s),      valid range: 0.5 to 30.3
rH  : Relative humidity (%),         valid range: 5 to 100
oob : Out-of-bounds handling strategy:
        "nan"   – return NaN for any out-of-bounds row (defaul

## Test on the data from Brode et al

In [3]:
esm4 = pd.read_csv('C:\\Users\\nerc-user\\OneDrive - Nexus365\\UTCI_NN\\Main\\data\\ESM_4_Table_Offset.DAT', sep='\t', comment='*')
test = pd.read_csv('C:\\Users\\nerc-user\\OneDrive - Nexus365\\UTCI_NN\\Main\\data\\UTCI-Test-Data.txt', sep='\t', comment='#')

In [4]:
test['Tr'] = test['Tr-Ta'] + test['Ta']  # convert to absolute MRT

### Run model on the test data to check the output is as expected

In [6]:
test["Offset_NN"] = calculate_utci(Ta=test['Ta'], Tr=test['Tr'], va=test['va'], rH=test['rH'], oob='nan')

In [7]:
test_analysis = test.copy()
y_true = test_analysis['UTCI']
y_pred = test_analysis['UTCI_polynomial']
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(f"RMSE: {rmse:.3f} °C")

RMSE: 2.777 °C


### Test Clamping

In [14]:
calculate_utci(Ta=70, Tr=28, va=10, rH=100, oob='clamp')

np.float64(80.78778839111328)

In [13]:
calculate_utci(Ta=70, Tr=28, va=10, rH=110, oob='clamp')

np.float64(80.78778839111328)

### Input shape mismatch - should return an error

In [12]:
calculate_utci(Ta=np.array([20.0, 25.0]),
        Tr=np.array([25.0]),          # wrong length
        va=np.array([ 1.0,  2.0]),
        rH=np.array([50.0, 60.0]))

ValueError: All inputs must have the same length, got shapes: {1, 2}

### Single input as an array

In [11]:
calculate_utci(Ta=np.array([20.0]),
            Tr=np.array([25.0]),
            va=np.array([ 1.0]),
            rH=np.array([50.0]))

array([21.4264946])

### One valid row

#### Nan

In [10]:
calculate_utci(
            Ta=np.array([20.0,  999.0]),
            Tr=np.array([25.0, 1000.0]),
            va=np.array([ 1.0,    1.0]),
            rH=np.array([50.0,   50.0]),
            oob="nan")

array([21.4264946,        nan])

#### clamping

In [9]:
calculate_utci(
            Ta=np.array([20.0,  999.0]),
            Tr=np.array([25.0, 1000.0]),
            va=np.array([ 1.0,    1.0]),
            rH=np.array([50.0,   50.0]),
            oob="clamp")

array([21.4264946 , 76.80784607])

## Edge cases, One valid, one out of bounds output

In [8]:
calculate_utci(
            Ta=np.array([-50.0, 50.0]),
            Tr=np.array([0.0, 135.0]),
            va=np.array([  1.0,   35.0]),
            rH=np.array([ 50.0,  50.0]),
            oob="nan",
        )

array([-42.43745804,          nan])